# Üniversite kapısında kadınlar erkekleri ne zaman geçti?
Veri: Dünya Bankası, brüt yükseköğretim okullaşma oranı (SE.TER.ENRR.FE ve .MA). Çalıştırma: bu klasörden "Restart & Run All".

## 1. Veri yükleme

In [1]:
import pandas as pd, glob
def load(code):
    f = glob.glob(f'../data/raw/API_{code}_*.csv')[0]
    d = pd.read_csv(f, skiprows=4)
    yrs = [c for c in d.columns if c.isdigit()]
    return d.melt(id_vars=['Country Name','Country Code'], value_vars=yrs, var_name='year', value_name=code)
fe, ma = load('SE.TER.ENRR.FE'), load('SE.TER.ENRR.MA')
df = fe.merge(ma, on=['Country Name','Country Code','year'])
df.columns = ['name','code','year','female','male']
meta = pd.read_csv(glob.glob('../data/raw/Metadata_Country_*.csv')[0])[['Country Code','Region']]
meta.columns = ['code','region']
df = df.merge(meta, on='code')
print('Ham satır:', len(df))

Ham satır: 17490


## 2. Temizleme
Kararlar: (a) Bölge bilgisi boş olanlar ülke değil "dünya, gelir grubu" gibi toplamlardır, çıkarıldı. (b) Kadın veya erkek değeri eksik satırlar çıkarıldı, çünkü karşılaştırma yapılamaz. (c) Erkek oranı 0 olan satırlar çıkarıldı.

In [2]:
df = df[df.region.notna()]
df = df.dropna(subset=['female','male']); df = df[df.male > 0]
df['year'] = df.year.astype(int)
df['name'] = df.name.str.replace(',', ';'); df['region'] = df.region.str.replace(',', ';')
print('Temiz satır:', len(df), '| ülke:', df.code.nunique(), '| yıllar:', df.year.min(), '-', df.year.max())

Temiz satır: 4589 | ülke: 203 | yıllar: 1970 - 2025


## 3. Keşifsel analiz

In [3]:
df['women_more'] = df.female > df.male
share = df.groupby('year').women_more.mean().mul(100).round(1)
n = df.groupby('year').size()
print(share[n >= 50].iloc[::5])
tur = df[df.code == 'TUR'].sort_values('year')
print(tur[['year','female','male']].tail(12))

year
1971    10.5
1976     9.6
1981    25.0
1986    25.9
2001    62.5
2006    63.8
2011    64.2
2016    77.6
2021    84.7
Name: women_more, dtype: float64
       year      female        male
14023  2012   65.121498   74.876648
14288  2013   74.183250   84.886978
14553  2014   81.519478   92.242172
14818  2015   89.853310  101.024689
15083  2016   97.503853  109.631813
15348  2017  104.269279  116.501297
15613  2018  109.953590  120.712044
15878  2019  114.014442  119.851822
16143  2020  117.806450  119.907112
16408  2021  126.784470  124.792427
16673  2022  130.030579  125.242462
16938  2023  109.999863  101.941071


## 4. Sitede kullanılan özet dosyanın dışa aktarımı

In [4]:
import os
os.makedirs('../site/data', exist_ok=True); os.makedirs('../data/processed', exist_ok=True)
out = df[['code','name','region','year','female','male']].round(2)
out.to_csv('../site/data/gap.csv', index=False); out.to_csv('../data/processed/gap.csv', index=False)
print('Yazıldı:', len(out), 'satır')

Yazıldı: 4589 satır
